# Data Cleaning and Preprocessing
This notebook performs Step 3 of the Real Estate ML pipeline: cleaning the raw dataset scraped in Step 2. We handle missing values, duplicates, normalize strings, clean numerical values (removing strings like 'Sq.ft', '₹', etc.), handle outliers, and create useful features.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Inspect the Dataset
First, we load the raw dataset and inspect its structure, data types, and check for missing values and duplicates.

In [2]:
df = pd.read_csv('../data/raw/property_data.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names: {list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values count:\n{df.isnull().sum()}")
print(f"\nDuplicate count: {df.duplicated().sum()}")

display(df.head())

Dataset shape: (250, 8)

Column names: ['Property_Name', 'Location', 'Property_Type', 'Price_INR', 'Area_sqft', 'BHK', 'Bathrooms', 'Listing_URL']

Data types:
Property_Name     object
Location          object
Property_Type     object
Price_INR        float64
Area_sqft        float64
BHK                int64
Bathrooms          int64
Listing_URL       object
dtype: object

Missing values count:
Property_Name     0
Location          0
Property_Type     0
Price_INR         0
Area_sqft        11
BHK               0
Bathrooms         0
Listing_URL       0
dtype: int64

Duplicate count: 0


,Property_Name,Location,Property_Type,Price_INR,Area_sqft,BHK,Bathrooms,Listing_URL
0,Luxury 4 BHK in South Mumbai,Lower Parel,Apartment,53100000.0,1479.0,2,2,https://www.sample-realestate.com/mumbai/prope...
1,Luxury 5 BHK in Borivali West,Lower Parel,Apartment,63100000.0,2616.0,3,3,https://www.sample-realestate.com/mumbai/prope...
2,Luxury 2 BHK in Juhu,Bandra East,Apartment,6700000.0,3578.0,3,4,https://www.sample-realestate.com/mumbai/prope...
3,Luxury 3 BHK in South Mumbai,Juhu,Apartment,87900000.0,2579.0,2,2,https://www.sample-realestate.com/mumbai/prope...
4,Luxury 5 BHK in Powai,Worli,Apartment,93600000.0,1162.0,1,2,https://www.sample-realestate.com/mumbai/prope...


## 2. Handle Duplicate Records
We remove duplicate listings as they can skew model training and analysis. Because the scraped dataset can contain the exact same properties multiple times (due to pagination or overlapping search results), we use `.drop_duplicates()`.

In [3]:
before_dup = len(df)
df = df.drop_duplicates()
after_dup = len(df)

print(f"Before duplicates: {before_dup}")
print(f"After duplicates: {after_dup}")
print(f"Duplicates removed: {before_dup - after_dup}")

Before duplicates: 250
After duplicates: 250
Duplicates removed: 0


## 3. Handle Missing Values
We fill missing Area_sqft and Bathrooms based on the medians of their respective property types and BHK. This is much more accurate than filling with 0 or the overall mean, since a 4BHK Villa will naturally have more area/bathrooms than a 1BHK Apartment.

In [4]:
# Fill Area_sqft using the median area for that specific Property_Type and BHK
df['Area_sqft'] = df.groupby(['Property_Type', 'BHK'])['Area_sqft'].transform(lambda x: x.fillna(x.median()))
# If any still remain (due to unique combinations), fill with overall median
df['Area_sqft'] = df['Area_sqft'].fillna(df['Area_sqft'].median())

# Fill Bathrooms using the median bathrooms for that BHK
df['Bathrooms'] = df.groupby('BHK')['Bathrooms'].transform(lambda x: x.fillna(x.median()))
# Fallback: assume Bathrooms = BHK + 1 if still missing
df['Bathrooms'] = df['Bathrooms'].fillna(df['BHK'] + 1)

print(f"Remaining missing values:\n{df.isnull().sum()}")

Remaining missing values:
Property_Name    0
Location         0
Property_Type    0
Price_INR        0
Area_sqft        0
BHK              0
Bathrooms        0
Listing_URL      0
dtype: int64


## 4 & 5. Clean Numerical & BHK Columns
We defensively clean the `Price_INR`, `Area_sqft`, and `BHK` columns. Even if they are already numeric, raw scraped data often contains strings like '1.25 Cr', '1500 sq.ft', or '2 BHK'. We extract the numbers using regular expressions.

In [5]:
import re

# Defensively extract floats for numeric columns if they are strings
if df['Price_INR'].dtype == object:
    df['Price_INR'] = df['Price_INR'].astype(str).str.replace(',', '').str.extract(r'(\d+\.?\d*)').astype(float)
    
if df['Area_sqft'].dtype == object:
    df['Area_sqft'] = df['Area_sqft'].astype(str).str.replace(',', '').str.extract(r'(\d+\.?\d*)').astype(float)

if df['BHK'].dtype == object:
    df['BHK'] = df['BHK'].astype(str).str.extract(r'(\d+)').astype(float)

print("Numerical columns cleaned!")

Numerical columns cleaned!


## 6 & 7. Clean Location and Property Type Data
We standardize the text columns to Title Case and strip trailing whitespaces to ensure consistency (e.g., 'Andheri West ' becomes 'Andheri West').

In [6]:
df['Location'] = df['Location'].astype(str).str.strip().str.title()
df['Property_Type'] = df['Property_Type'].astype(str).str.strip().str.title()
print("Categorical columns standardized.")

Categorical columns standardized.


## 8. Outlier Analysis
We remove extreme values that are likely errors. For instance, areas under 100 sqft or unrealistic price per sqft values (outside the 1st to 99th percentile).

In [7]:
initial_len = len(df)

# 1. Area threshold
df = df[df['Area_sqft'] >= 100]

# 2. Bathroom threshold
df = df[df['Bathrooms'] <= 10]

# 3. Price per sqft threshold
temp_price_sqft = df['Price_INR'] / df['Area_sqft']
lower_bound = temp_price_sqft.quantile(0.01)
upper_bound = temp_price_sqft.quantile(0.99)
df = df[(temp_price_sqft >= lower_bound) & (temp_price_sqft <= upper_bound)]

print(f"Removed {initial_len - len(df)} rows due to outliers.")

Removed 6 rows due to outliers.


## 9. Feature Creation
Creating the `price_per_sqft` feature. This is extremely useful for ML models predicting real estate prices as it captures the intrinsic value of the location.

In [8]:
df['price_per_sqft'] = (df['Price_INR'] / df['Area_sqft']).round(2)
print("Feature 'price_per_sqft' created successfully.")

Feature 'price_per_sqft' created successfully.


## 10. Remove Irrelevant Columns
Columns like `Property_Name` and `Listing_URL` have unique high-cardinality values that aren't useful for model training. We drop them here but keep `Location` for mapping.

In [9]:
cols_to_drop = ['Listing_URL', 'Property_Name']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])
print(f"Dropped columns: {cols_to_drop}")

Dropped columns: ['Listing_URL', 'Property_Name']


## 11 & 12. Final Validation and Saving Dataset
We confirm the data types and null counts before saving to the processed directory.

In [10]:
print("\n--- Final Dataset Validation ---")
print(f"Final dataset shape: {df.shape}")
print(f"Final columns: {list(df.columns)}")
print(f"\nFinal data types:\n{df.dtypes}")
print(f"\nRemaining missing values:\n{df.isnull().sum()}")

output_path = '../data/processed/cleaned_property_data.csv'
df.to_csv(output_path, index=False)
print(f"\nSaved cleaned dataset to: {output_path}")


--- Final Dataset Validation ---
Final dataset shape: (244, 7)
Final columns: ['Location', 'Property_Type', 'Price_INR', 'Area_sqft', 'BHK', 'Bathrooms', 'price_per_sqft']

Final data types:
Location           object
Property_Type      object
Price_INR         float64
Area_sqft         float64
BHK                 int64
Bathrooms           int64
price_per_sqft    float64
dtype: object

Remaining missing values:
Location          0
Property_Type     0
Price_INR         0
Area_sqft         0
BHK               0
Bathrooms         0
price_per_sqft    0
dtype: int64

Saved cleaned dataset to: ../data/processed/cleaned_property_data.csv
